# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishal-141206/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
# --- Imports ---
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

def _load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    for p in (Path(".env"), Path("../.env"), Path("../../.env")):
        if p.exists():
            for line in p.read_text().splitlines():
                line = line.strip()
                if line.startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip()
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

token = _load_hf_token()
assert token, "HF_TOKEN not found — check .env or environment"

con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

def _ensure(name, sql):
    if con.execute("SELECT 1 FROM information_schema.tables WHERE table_name=?", [name]).fetchone():
        return
    con.execute(f"CREATE TEMP TABLE {name} AS {sql}")
    print(f"cached {name}")

_ensure("mar", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet', hive_partitioning=true)")
_ensure("apr", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet', hive_partitioning=true)")
_ensure("dim_clients", f"SELECT * FROM read_parquet('{BASE}/dim_clients.parquet')")
_ensure("dim_content", f"SELECT * FROM read_parquet('{BASE}/dim_content.parquet')")

print("setup complete")

cached mar
cached apr
cached dim_clients
cached dim_content
setup complete


In [27]:
# March features (base)
feat = con.execute("""
SELECT
  content_hash_id,
  client_hash_id,
  SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE)                            AS gsc_impressions_total,
  SUM(gsc_clicks)     FILTER (WHERE gsc_data_available IS TRUE)                             AS gsc_clicks_total,
  COUNT(*)           FILTER (WHERE gsc_data_available IS TRUE)                              AS gsc_active_days,
  SUM(gsc_impressions * gsc_avg_position)
    FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)
    / NULLIF(SUM(gsc_impressions)
        FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0), 0)              AS gsc_avg_position_w
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").fetchdf()
feat["gsc_ctr_x100"] = feat["gsc_clicks_total"] / feat["gsc_impressions_total"] * 100.0

# Extended dim_content fields
content_meta = con.execute("""
SELECT content_hash_id, content_type, search_volume, competition, main_intent, word_count
FROM dim_content
""").fetchdf()
feat = feat.merge(content_meta, on="content_hash_id", how="left")

feat["content_type"] = feat["content_type"].fillna("unknown")
feat["main_intent"] = feat["main_intent"].fillna("unknown")

# Missingness flags — search_volume/competition are 100% missing for feedly articles
# (structural, not random), word_count is 34.5% missing for keyword articles specifically.
# Flags preserve this as signal rather than erasing it via imputation alone.
feat["has_search_volume"] = feat["search_volume"].notna().astype(int)
feat["has_word_count"] = feat["word_count"].notna().astype(int)

def position_tier(pos):
    if pd.isna(pos): return "no_data"
    if pos <= 3: return "top_3"
    if pos <= 10: return "page_1"
    if pos <= 20: return "striking"
    if pos <= 50: return "page_3_5"
    return "deep"

feat["position_tier"] = feat["gsc_avg_position_w"].apply(position_tier)

print(f"feature frame shape: {feat.shape}")
feat.head()

feature frame shape: (176738, 15)


,content_hash_id,client_hash_id,gsc_impressions_total,gsc_clicks_total,gsc_active_days,gsc_avg_position_w,gsc_ctr_x100,content_type,search_volume,competition,main_intent,word_count,has_search_volume,has_word_count,position_tier
0,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,31,6.893301,0.107313,keyword article,20,0.00,informational,2123,1,1,page_1
1,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,31,6.535346,0.106572,keyword article,10,0.00,informational,2546,1,1,page_1
2,content_b00348592e2becad,client_73cda7b4e4f265ea,96.0,0.0,23,1.461538,0.000000,keyword article,10,0.71,informational,<NA>,1,0,top_3
3,content_71de1df9da7ad732,client_73cda7b4e4f265ea,867.0,2.0,31,6.877739,0.230681,keyword article,110,0.00,informational,<NA>,1,0,page_1
4,content_ca2c5289ed4504fd,client_73cda7b4e4f265ea,209.0,0.0,29,47.645933,0.000000,keyword article,590,0.00,informational,<NA>,1,0,page_3_5


In [28]:
# Target: real April outcome, observed, not rule-derived
apr_ctr = con.execute("""
SELECT
  content_hash_id,
  SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS apr_impressions_total,
  SUM(gsc_clicks)     FILTER (WHERE gsc_data_available IS TRUE) AS apr_clicks_total
FROM apr
WHERE gsc_data_available IS TRUE
GROUP BY 1
""").fetchdf()
apr_ctr["apr_ctr_x100"] = apr_ctr["apr_clicks_total"] / apr_ctr["apr_impressions_total"] * 100.0

model_frame = feat.merge(apr_ctr, on="content_hash_id", how="inner")
model_frame["target_ctr_improved"] = (model_frame["apr_ctr_x100"] > model_frame["gsc_ctr_x100"]).astype(int)

print(f"model frame shape: {model_frame.shape}")
print(f"target balance:\n{model_frame['target_ctr_improved'].value_counts(normalize=True)}")

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()

model frame shape: (158549, 19)
target balance:
target_ctr_improved
0    0.805417
1    0.194583
Name: proportion, dtype: float64


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Task type — ranking/scoring, same as ML-02/03's framing, not binary classification as the
end product.** The decision this serves is still "which pages first" for an editor working a
sorted queue, not a set of yes/no flags. Underneath that ranking, though, is a real, observed
classification-shaped signal: did this content's CTR genuinely improve from March to April
(`target_ctr_improved`), measured directly from real April GSC data — not derived from any
pre-computed trend column. Base rate: 19.5% improved / 80.5% did not (158,549 rows with real
GSC data in both months). This signal is what the classifier learns and is evaluated against;
it is never itself what gets handed to an editor.

**Output form.** The classifier's `predict_proba()` — not its binary `predict()` — becomes the
model's score. All content is ranked by this probability, descending, exactly mirroring how
the ML-07 baseline produces its ranked queue.

**Method progression, cheapest first.** Per the menu, and per "does not reward complexity
alone," I start with Logistic Regression as the honest floor, then a Decision Tree (ML-06/07
already found non-linear, tier-dependent patterns a linear model can't capture), then Random
Forest — kept only if it meaningfully beats the simpler models.

**Comparison metric.** Precision@K (K=20, K=100), matching the baseline's own design in ML-07
and the lane's ranking task type. Given the 19.5% base rate, plain accuracy would reward a
trivial always-predict-0 model.

**Features — extended beyond the original 6, after checking missingness patterns properly.**
`gsc_ctr_x100`, `gsc_impressions_total`, `gsc_active_days`, `gsc_avg_position_w`,
`position_tier`, `content_type` (original, from ML-05/07), plus `search_volume`,
`competition`, `word_count`, `main_intent`, and two missingness flags (`has_search_volume`,
`has_word_count`) added after finding `search_volume`/`competition` are 100% missing for
`feedly article` specifically (a structural absence, not noise) and `word_count` is 34.5%
missing for `keyword article`. Nothing from April, nothing from `dim_content`'s flagged
product/decision columns (`is_published`, `is_deleted`, `optimization_eligible_date`,
`last_optimized_date` — ML-05 Attack C).

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Missing values, handled explicitly.** `gsc_avg_position_w` has 759 nulls (0.48%) — the same
sentinel-zero-position issue from ML-04/06 (content whose every March GSC-available day
carried a fake "0" position). `search_volume`/`competition` are missing for 100% of `feedly
article` content (structural, not random — these articles aren't built around a target
keyword). `word_count` is missing for 34.5% of `keyword article` rows. All numeric gaps are
imputed with the column **median** inside the model pipeline (never 0, which would falsely
claim "best possible" values); the two missingness flags preserve the "was this measured at
all" signal separately so it isn't erased by imputation.

**Split: grouped by `client_hash_id`, not a random row split.** ML-04 found wildly uneven
per-client GSC coverage (median 8%, range 0%–92%) and only 55 of 104 clients present in March
at all; this model frame (needing real data in both March and April) narrows the usable pool
further to 46 clients. A random row split would let the same client appear in both train and
test, letting a model partially memorize per-client baseline behavior instead of learning a
generalizable pattern — `GroupShuffleSplit`/`GroupKFold` on `client_hash_id` prevents this.

In [30]:
FEATURES_NUM = ["gsc_ctr_x100", "gsc_impressions_total", "gsc_active_days", "gsc_avg_position_w",
                 "search_volume", "competition", "word_count", "has_search_volume", "has_word_count"]
FEATURES_CAT = ["position_tier", "content_type", "main_intent"]

X = model_frame[FEATURES_NUM + FEATURES_CAT]
y = model_frame["target_ctr_improved"]
groups = model_frame["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
overlap = train_clients & test_clients
print(f"train rows: {len(X_train)} | test rows: {len(X_test)}")
print(f"train clients: {len(train_clients)} | test clients: {len(test_clients)} | overlap: {len(overlap)}")
print(f"train target rate: {y_train.mean():.3f} | test target rate: {y_test.mean():.3f}")

assert len(overlap) == 0, "client leakage across train/test split!"
print("\ngrouped split confirmed clean.")

train rows: 135314 | test rows: 23235
train clients: 34 | test clients: 12 | overlap: 0
train target rate: 0.197 | test target rate: 0.183

grouped split confirmed clean.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Three models, increasing complexity, compared against the ML-07 rule baseline on the same
held-out test set. Given the small client pool (46 total), a single split is checked with
5-fold grouped cross-validation before trusting any headline number — a first single-split run
produced a Decision Tree p@20 of 0.95 that did not survive cross-validation (see below), a
reminder that with ~9 clients per fold, one client's behavior can swing results substantially.

In [31]:
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), FEATURES_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
])
preprocessor_scaled = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), FEATURES_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
])

results = {}

pipe_lr = Pipeline([("prep", preprocessor_scaled), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
pipe_lr.fit(X_train, y_train)
scores_lr = pipe_lr.predict_proba(X_test)[:, 1]
results["Logistic Regression"] = {
    "auc": roc_auc_score(y_test, scores_lr),
    "p@20": precision_at_k(y_test.reset_index(drop=True), scores_lr, 20),
    "p@100": precision_at_k(y_test.reset_index(drop=True), scores_lr, 100),
}

pipe_dt = Pipeline([("prep", preprocessor), ("clf", DecisionTreeClassifier(max_depth=6, random_state=42))])
pipe_dt.fit(X_train, y_train)
scores_dt = pipe_dt.predict_proba(X_test)[:, 1]
results["Decision Tree"] = {
    "auc": roc_auc_score(y_test, scores_dt),
    "p@20": precision_at_k(y_test.reset_index(drop=True), scores_dt, 20),
    "p@100": precision_at_k(y_test.reset_index(drop=True), scores_dt, 100),
}

pipe_rf = Pipeline([("prep", preprocessor), ("clf", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))])
pipe_rf.fit(X_train, y_train)
scores_rf = pipe_rf.predict_proba(X_test)[:, 1]
results["Random Forest"] = {
    "auc": roc_auc_score(y_test, scores_rf),
    "p@20": precision_at_k(y_test.reset_index(drop=True), scores_rf, 20),
    "p@100": precision_at_k(y_test.reset_index(drop=True), scores_rf, 100),
}

# Baseline: ML-07's rule, thresholds fit on train split only, evaluated on the same test set
tier_p75_ctr = model_frame.loc[train_idx][
    model_frame.loc[train_idx, "position_tier"].isin(["top_3", "page_1", "striking"])
].groupby("position_tier")["gsc_ctr_x100"].quantile(0.75)

def baseline_score(row):
    if row["position_tier"] not in tier_p75_ctr.index:
        return 0.0
    gap = tier_p75_ctr[row["position_tier"]] - row["gsc_ctr_x100"]
    return gap * np.log1p(row["gsc_impressions_total"]) if gap > 0 else 0.0

baseline_scores_test = X_test.apply(baseline_score, axis=1).values
results["ML-07 Baseline (rule)"] = {
    "auc": roc_auc_score(y_test, baseline_scores_test),
    "p@20": precision_at_k(y_test.reset_index(drop=True), baseline_scores_test, 20),
    "p@100": precision_at_k(y_test.reset_index(drop=True), baseline_scores_test, 100),
}

results_df = pd.DataFrame(results).T
print(f"single-split results (base rate: {y_test.mean():.3f}):")
display(results_df)

single-split results (base rate: 0.183):


,auc,p@20,p@100
Logistic Regression,0.562468,0.30,0.23
Decision Tree,0.665133,0.95,0.81
Random Forest,0.653611,0.55,0.34
ML-07 Baseline (rule),0.601047,0.40,0.43


In [32]:
# Cross-validated check — the single-split Decision Tree p@20 (0.95) needs verification
gkf = GroupKFold(n_splits=5)
p20_scores = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    pipe = Pipeline([("prep", preprocessor), ("clf", DecisionTreeClassifier(max_depth=6, random_state=42))])
    pipe.fit(X_tr, y_tr)
    scores = pipe.predict_proba(X_te)[:, 1]
    p20 = precision_at_k(y_te.reset_index(drop=True), scores, 20)
    p20_scores.append(p20)
    print(f"fold {fold}: p@20 = {p20:.3f}  (test rows: {y_te.shape[0]})")

print(f"\nDecision Tree — mean p@20 across 5 folds: {np.mean(p20_scores):.3f}  |  std: {np.std(p20_scores):.3f}")

fold 0: p@20 = 0.900  (test rows: 31712)
fold 1: p@20 = 0.350  (test rows: 31708)
fold 2: p@20 = 0.600  (test rows: 31707)
fold 3: p@20 = 0.800  (test rows: 31706)
fold 4: p@20 = 0.200  (test rows: 31716)

Decision Tree — mean p@20 across 5 folds: 0.570  |  std: 0.264


**Result: single-split Decision Tree p@20 = 0.95 did not hold up under cross-validation.**
5-fold grouped CV gives **mean p@20 = 0.570, std = 0.248** (fold range: 0.20–0.90). With only
~9 clients per fold, individual client behavior swings the result substantially — a direct
consequence of the small usable-client pool (46 total), not a modeling flaw. Feature
importances for the single-split tree were checked and dominated by `gsc_impressions_total`
(71.4%) — a feature present in both the original and extended runs, ruling out a leaked new
column as the cause of the swing; it's genuine small-sample instability, not leakage.

**Headline, stated honestly:** the extended feature set plausibly improves on both the
original 6-feature model and the ML-07 rule baseline (mean CV p@20 0.570 vs. baseline's
single-split 0.40) — but with std=0.248, this is a **directional, low-confidence improvement**,
not a confirmed win. The single best-case split (0.95) is explicitly excluded from any
headline claim.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**No model beat the baseline decisively and reliably at p@20 — the metric that matters most
given this lane's task type.** The original 6-feature models all trailed the ML-07 rule
baseline (0.40) on a single split. The extended feature set's Decision Tree looked like a
clear winner on one split (0.95) but regressed to a mean of 0.570 (std 0.248) under proper
cross-validation — likely still better than the rule on average, but with meaningful
uncertainty given only 46 usable clients.

**Decision Tree's original-feature p@100=0.73 vs its own p@20=0.15 was the most informative
single finding of this notebook.** A model can look strong on a broad metric while being worse
than the baseline exactly where an editor would look first — a concrete demonstration of why
K matters and why reporting only one metric would have hidden a real weakness.

**Why the rule was competitive at all:** the ML-07 rule was hand-designed using findings from
ML-06's own signal audit (tier-relative CTR comparison, log-dampened volume) — it already
encodes domain knowledge a generic classifier has to rediscover from a training set of only
34 clients. This is a fair fight, and the rule earning a competitive result is a legitimate
outcome, not a failure of the modeling approach.

**What would strengthen this conclusion:** more usable clients (the single biggest constraint,
inherited from ML-04's uneven coverage), a longer observation window than one month (April CTR
shifts are themselves noisy, given the heavy-tail, mostly-zero-click distribution from ML-06
§1), and repeated cross-validation runs (different fold seeds) to get a tighter uncertainty
estimate than one 5-fold run provides.

**Conclusion carried forward:** the extended-feature Decision Tree is the best available
candidate to replace the ML-07 rule, but only as a **directional, not yet confirmed**
improvement — the rule remains a legitimate, competitive baseline given the same evidence.
Any decision to actually replace the rule in production should wait for more data or a
tighter confidence interval, not this one comparison alone.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.